In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv('/workspaces/Major_project/data/raw/districtwise-crime-against-women-2017-onwards.csv')
print("Loaded:", df.shape)

Loaded: (6281, 49)


In [12]:
df=df.fillna(0)

In [15]:
df['district_name']=df['district_name'].str.strip().str.title()  # Remove leading/trailing whitespace from district names
df['state_name']=df['state_name'].str.strip().str.title()  # Remove leading/trailing whitespace from state names

In [16]:
id_columns=['id','year','state_name','state_code','district_name','district_code','registration_circles']
crime_columns = [c for c in df.columns if c not in id_columns]

print(f"Total crime columns: {len(crime_columns)}")
print(f"First 5: {crime_columns[:5]}")
print(f"Last 5: {crime_columns[-5:]}")

Total crime columns: 42
First 5: ['murder_with_rape_gang_rape', 'dowry_deaths', 'abetment_to_suicide_of_women', 'miscarriage', 'acid_attack']
Last 5: ['child_sexual_harassment', 'use_of_child_for_pornography', 'offences_of_pocso_act', 'pocso_act_unnatural_offences', 'indecent_representation_of_women']


In [17]:
# ============================================================
# SEVERITY SCORING
# ============================================================
# Assign a severity weight to each crime type.
# Higher weight = more severe crime.
# Scale: 1 (least severe) to 5 (most severe)

severity_weights = {
    # Severity 5 - Extreme
    'murder_with_rape_gang_rape': 5,
    'rape_women_above_18': 5,
    'rape_girls_below_18': 5,
    'child_rape': 5,
    'acid_attack': 5,

    # Severity 4 - Very High
    'attempt_to_acid_attack': 4,
    'attempt_to_commit_rape_above_18': 4,
    'attempt_to_commit_rape_girls_below_18': 4,
    'human_trafficking': 4,
    'kidnapping_abduction_in_order_to_murder': 4,

    # Severity 3 - High
    'dowry_deaths': 3,
    'abetment_to_suicide_of_women': 3,
    'kidnapping_for_ransom': 3,
    'kidnp_and_abductn_of_women_above_18_for_marrg': 3,
    'kidnp_and_abductn_of_girls_below_18_for_marrg': 3,
    'procuration_of_minor_girls': 3,
    'selling_of_minor_girls': 3,
    'buying_of_minor_girls': 3,
    'sexual_assault_of_children': 3,
    'child_sexual_harassment': 3,
    'use_of_child_for_pornography': 3,
    'offences_of_pocso_act': 3,
    'pocso_act_unnatural_offences': 3,
    'indecent_representation_of_women': 3,
    'publish_transmit_sexually_explicit_mtrl': 3,

    # Severity 2 - Medium
    'cruelty_by_husband_or_his_relatives': 2,
    'kidnapping_and_abduction': 2,
    'kidnapping_and_abduction_of_women_others': 2,
    'assault_on_womenabove_18': 2,
    'assault_on_women_below_18': 2,
    'insult_to_the_modesty_of_women_above_18': 2,
    'insult_to_the_modesty_of_women_below_18': 2,
    'other_women_centric_cyber_crimes': 2,
    'protection_of_women_from_domestic_violence': 2,

    # Severity 1 - Lower
    'miscarriage': 1,
    'importation_of_girls_from_foreign': 1,
    'dowry_prohibition': 1,
    'inducing_child_for_prostitution': 1,
    'detain_prsn_in_prostitution_premises': 1,
    'prostitution_near_public_places': 1,
    'seducing_soliciting_for_prostitution': 1,
    'other_sections_under_itp_act': 1,
}

# Verify all crime columns have a weight
missing_weights = [c for c in crime_columns if c not in severity_weights]
if missing_weights:
    print(f"⚠️  WARNING: These columns have no severity weight: {missing_weights}")
else:
    print("✅ All crime columns have severity weights assigned.")

# 4. Calculate severity-weighted score for each row
df['severity_score'] = sum(
    df[col] * severity_weights.get(col, 1) for col in crime_columns
)

# 5. Calculate raw total crimes (unweighted, for reference)
df['total_crimes'] = df[crime_columns].sum(axis=1)

print("\nSample severity scores (first 10 rows):")
print(df[['district_name', 'year', 'total_crimes', 'severity_score']].head(10))

✅ All crime columns have severity weights assigned.

Sample severity scores (first 10 rows):
              district_name  year  total_crimes  severity_score
0                  Nicobars  2017           8.0            30.0
1  North And Middle Andaman  2017          53.0           193.0
2            South Andamans  2017          71.0           238.0
3             Ananthapuramu  2017        1100.0          2469.0
4             Ananthapuramu  2017          14.0            34.0
5                  Chittoor  2017         505.0          1129.0
6             East Godavari  2017        1410.0          3210.0
7             East Godavari  2017         351.0           799.0
8                    Guntur  2017        1544.0          3546.0
9                    Guntur  2017         889.0          1978.0


In [24]:
from pathlib import Path

# ============================================================
# AGGREGATE TO DISTRICT LEVEL
# ============================================================

# Group by year, state, district and sum everything
district_data = df.groupby(
    ['year', 'state_name', 'state_code', 'district_name', 'district_code'],
    as_index=False
)[crime_columns + ['total_crimes', 'severity_score']].sum()

print(f"Original rows: {len(df)}")
print(f"Aggregated rows: {len(district_data)}")
print(f"Districts: {district_data['district_name'].nunique()}")
print(f"States: {district_data['state_name'].nunique()}")

print("\nSample aggregated data:")
print(district_data.head(10))

# Save both versions
output_dir = Path('/workspaces/Major_project/data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(output_dir / 'cleaned_crime_data.csv', index=False)
district_data.to_csv(output_dir / 'district_level_data.csv', index=False)

print("\n✅ Saved:")
print("   - data/processed/cleaned_crime_data.csv")
print("   - data/processed/district_level_data.csv")

Original rows: 6281
Aggregated rows: 5104
Districts: 800
States: 36

Sample aggregated data:
   year                   state_name  state_code             district_name  \
0  2017  Andaman And Nicobar Islands          35                  Nicobars   
1  2017  Andaman And Nicobar Islands          35  North And Middle Andaman   
2  2017  Andaman And Nicobar Islands          35            South Andamans   
3  2017               Andhra Pradesh          28             Ananthapuramu   
4  2017               Andhra Pradesh          28                  Chittoor   
5  2017               Andhra Pradesh          28             East Godavari   
6  2017               Andhra Pradesh          28                    Guntur   
7  2017               Andhra Pradesh          28                   Krishna   
8  2017               Andhra Pradesh          28                   Kurnool   
9  2017               Andhra Pradesh          28                       Ntr   

   district_code  murder_with_rape_gang_rape  do